# Regression Walkthrough: Wine Quality

Same overall shape as [`02_classification_bank_marketing.ipynb`](02_classification_bank_marketing.ipynb),
applied to a regression task, using the UCI **Wine Quality** dataset — 6,497 physicochemical
measurements (acidity, sugar, sulfur dioxide, alcohol, etc.) on Portuguese "Vinho Verde" wine
samples, with `quality` (a 0-10 sensory score) as the target.

This notebook assumes you've read `02` already — it focuses on what's *different* for
regression rather than re-explaining everything from scratch. See [`README.md`](README.md)
for the full notebook index.

In [1]:
!pip install -q dscompanion[notebooks]

zsh:1: no matches found: dscompanion[notebooks]


In [2]:
from pathlib import Path
import pandas as pd
from ucimlrepo import fetch_ucirepo

cache_dir = Path(".data_cache")
cache_dir.mkdir(exist_ok=True)
data_path = cache_dir / "wine_quality.parquet"

if not data_path.exists():
    ds = fetch_ucirepo(id=186)
    df = ds.data.features.copy()
    df["quality"] = ds.data.targets["quality"]
    df.to_parquet(data_path, index=False)
else:
    df = pd.read_parquet(data_path)

print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df["quality"].describe()

6497 rows, 12 columns


count    6497.000000
mean        5.818378
std         0.873255
min         3.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         9.000000
Name: quality, dtype: float64

## Config and run

`quality` is a continuous score, not a class label, so `model.task="regression"`.
`split.method="random"` is used here rather than `"stratified"` — stratified splitting
is a classification concept (it preserves class proportions), which doesn't apply to a
continuous target.

In [3]:
from dscompanion.pipeline import PipelineConfig, PipelineRunner

cfg = PipelineConfig(
    name="wine_quality_regression",
    data={"path": str(data_path), "format": "parquet", "target": "quality"},
    split={"method": "random", "test_size": 0.2, "val_size": 0.1},
    model={"task": "regression", "algorithm": "xgboost"},
    reporting={"output_dir": "reports", "html_report": True},
)
result = PipelineRunner(cfg).run()
result.metrics.pivot(index="metric", columns="split", values="value")

2026-09-13 11:20:05  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:05  INFO      dscompanion.pipeline.runner  PipelineRunner  |  wine_quality_regression  v1.0


2026-09-13 11:20:05  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: regression  |  Algorithm: xgboost


2026-09-13 11:20:05  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:05  WARNING   dscompanion.pipeline.runner  Pre-flight: target.imbalance.strategy='class_weight' is only applicable to classification tasks. It will be ignored.


...

2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  Pipeline complete — 1.7s  |  Run: 20260913_112005


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  Report: reports/20260913_112005/reports/wine_quality_regression_v1.0_model_card.html


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  ============================================================


split,test,train,val
metric,,,
mae,0.491205,0.357836,0.481416
mape,0.087427,0.063154,0.087725
median_absolute_error,0.402094,0.281073,0.413567
r2,0.450055,0.717293,0.476101
rmse,0.637100,0.467092,0.621751


`result.metrics` keeps the same tidy `split`/`metric`/`value` shape used everywhere in
dscompanion — only the metric names change per task. For regression, `metric` takes
exactly five values: `rmse`, `mae`, `r2`, `mape`, and `median_absolute_error`.

## No leaderboard section here

`02` used `LeaderboardConfig(enabled=True)` to compare algorithms automatically. That
feature currently only supports `task="classification"` — `PipelineConfig` itself
rejects `leaderboard.enabled=True` combined with any other task at config-construction
time (a clear validation error, not a confusing crash mid-run). Run the cell below to
see it for yourself:

In [4]:
from pydantic import ValidationError

try:
    PipelineConfig(
        name="this_will_fail",
        data={"path": str(data_path), "format": "parquet", "target": "quality"},
        model={"task": "regression", "algorithm": "xgboost"},
        leaderboard={"enabled": True},
    )
except ValidationError as exc:
    print(exc)

1 validation error for PipelineConfig
  Value error, leaderboard.enabled=True currently requires model.task='classification', got 'regression' [type=value_error, input_value={'name': 'this_will_fail'...ard': {'enabled': True}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error


## Tuning — with a regression-appropriate metric

`TuningConfig`'s defaults (`metric="roc_auc"`, `direction="maximize"`) are shaped for
classification. `roc_auc` doesn't exist in a regression model's evaluation output at
all, so it must be overridden — here, to `rmse`. And unlike `roc_auc` (higher is
better), `rmse` is a loss: lower is better, so `direction="minimize"`.

In [5]:
cfg_tuned = PipelineConfig(
    name="wine_quality_regression_tuned",
    data={"path": str(data_path), "format": "parquet", "target": "quality"},
    split={"method": "random", "test_size": 0.2, "val_size": 0.1},
    model={"task": "regression", "algorithm": "xgboost"},
    tuning={"enabled": True, "n_trials": 20, "metric": "rmse", "direction": "minimize"},
    reporting={"output_dir": "reports", "html_report": False},
)
result_tuned = PipelineRunner(cfg_tuned).run()

print("Best params found:")
print(result_tuned.tuner.best_params_)
result_tuned.metrics.pivot(index="metric", columns="split", values="value")

2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  PipelineRunner  |  wine_quality_regression_tuned  v1.0


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: regression  |  Algorithm: xgboost


2026-09-13 11:20:07  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:07  WARNING   dscompanion.pipeline.runner  Pre-flight: target.imbalance.strategy='class_weight' is only applicable to classification tasks. It will be ignored.


...

2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  Pipeline complete — 41.5s  |  Run: 20260913_112007


2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  ============================================================


Best params found:
{'n_estimators': 437, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 0.000602521573620386, 'reg_lambda': 0.00019517224641449495}


split,test,train,val
metric,,,
mae,0.418853,0.005285,0.374364
mape,0.075279,0.000929,0.069096
median_absolute_error,0.286364,0.003600,0.224300
r2,0.473782,0.999930,0.561593
rmse,0.623205,0.007328,0.568763


## Explainability and the model card

Same API as classification — `permutation_enabled`/`shap_enabled` in `ExplainConfig`,
and `result.model_card.to_excel(...)`/`.to_html(...)` for the exported report. Nothing
regression-specific here; it's the same interface across every task.

In [6]:
cfg_explain = PipelineConfig(
    name="wine_quality_explain",
    data={"path": str(data_path), "format": "parquet", "target": "quality"},
    split={"method": "random", "test_size": 0.2, "val_size": 0.1},
    model={"task": "regression", "algorithm": "xgboost"},
    explain={"permutation_enabled": True},
    reporting={"output_dir": "reports", "html_report": False},
)
result_explain = PipelineRunner(cfg_explain).run()

display(result_explain.permutation_importance.importance_table())
fig = result_explain.permutation_importance.summary_plot()
fig.show()

2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  PipelineRunner  |  wine_quality_explain  v1.0


2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  Owner: unset  |  Task: regression  |  Algorithm: xgboost


2026-09-13 11:20:48  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:48  WARNING   dscompanion.pipeline.runner  Pre-flight: target.imbalance.strategy='class_weight' is only applicable to classification tasks. It will be ignored.


...

2026-09-13 11:20:49  INFO      dscompanion.pipeline.runner  ============================================================


2026-09-13 11:20:49  INFO      dscompanion.pipeline.runner  Pipeline complete — 1.2s  |  Run: 20260913_112048


2026-09-13 11:20:49  INFO      dscompanion.pipeline.runner  Report: n/a


2026-09-13 11:20:49  INFO      dscompanion.pipeline.runner  ============================================================


,feature,importance_mean,importance_std,rank
0,alcohol,0.268571,0.010546,1
1,free_sulfur_dioxide,0.136644,0.013185,2
2,volatile_acidity,0.135943,0.007945,3
3,sulphates,0.075187,0.007417,4
4,total_sulfur_dioxide,0.073496,0.004937,5
5,residual_sugar,0.064537,0.007116,6
6,density,0.048726,0.006281,7
7,chlorides,0.037808,0.004914,8
8,citric_acid,0.030304,0.004776,9
9,fixed_acidity,0.025339,0.003535,10


In [7]:
from pathlib import Path

Path("reports").mkdir(exist_ok=True)
result_explain.model_card.to_excel("reports/wine_quality_model_card.xlsx")
result_explain.model_card.to_html("reports/wine_quality_model_card.html")
print("Model card written to reports/")

2026-09-13 11:20:50  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 11 numeric, 0 categorical, 0 datetime, 0 boolean


2026-09-13 11:20:50  INFO      dscompanion.eda.univariate  UnivariateAnalyser fitted — 11 numeric, 0 categorical, 0 datetime, 0 boolean


2026-09-13 11:20:50  INFO      dscompanion.docs.model_card  ModelCard (xlsx) → reports/wine_quality_model_card.xlsx


/Users/dsnaveen/projects/domain-ml/src/dscompanion/docs/html_eda.py:535: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

2026-09-13 11:20:50  INFO      dscompanion.docs.model_card  ModelCard (html) → reports/wine_quality_model_card.html


Model card written to reports/


## Next

[`04_clustering_wholesale_customers.ipynb`](04_clustering_wholesale_customers.ipynb)
covers the last of dscompanion's three tasks: unsupervised clustering, where there's no
label to predict at all — and a config quirk worth understanding as a result.